# Customer Churn - Model Training & Evaluation
This notebook trains multiple ML models and a Neural Network, then compares their performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

import warnings
warnings.filterwarnings('ignore')

## 1. Load & Preprocess Data

In [ ]:
df = pd.read_csv('../data/customer_churn.csv')
df_model = pd.get_dummies(df.drop('CustomerID', axis=1), drop_first=True)

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training: {X_train.shape[0]} | Testing: {X_test.shape[0]}')

## 2. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, lr_pred):.4f}')
print(classification_report(y_test, lr_pred))

## 3. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, dt_pred):.4f}')
print(classification_report(y_test, dt_pred))

## 4. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, rf_pred):.4f}')
print(classification_report(y_test, rf_pred))

## 5. Neural Network (MLP)

In [ ]:
nn = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', max_iter=200, random_state=42)
nn.fit(X_train, y_train)
nn_pred = nn.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, nn_pred):.4f}')
print(classification_report(y_test, nn_pred))

## 6. Model Comparison

In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Neural Network']
accs = [accuracy_score(y_test, p) for p in [lr_pred, dt_pred, rf_pred, nn_pred]]

plt.figure(figsize=(10, 6))
colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']
bars = plt.bar(models, accs, color=colors, edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.4f}', ha='center', fontweight='bold', fontsize=12)
plt.ylim(0, 1.05)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.tight_layout()
plt.show()

## 7. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in [('Logistic Regression', lr), ('Decision Tree', dt), ('Random Forest', rf), ('Neural Network', nn)]:
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
predictions = [lr_pred, dt_pred, rf_pred, nn_pred]
for i, (name, pred) in enumerate(zip(models, predictions)):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 9. Save Best Model

In [ ]:
# Save the best model (Random Forest) and scaler for deployment
with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(rf, f)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model and scaler saved to models/ folder!')